# AI 기본법 XML 기반 Qdrant 테스트

기존 `qdrant_test.ipynb`의 Qdrant 흐름을 실제 법령 XML에 적용한 노트북입니다.

In [10]:
from pathlib import Path
import xml.etree.ElementTree as ET

from common.ai_model import get_embedding_model
from common.qdrant import get_qdrant_client
from qdrant_client.models import Distance, PointStruct, VectorParams

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
XML_PATH = PROJECT_ROOT / 'data' / 'ai_basic_law' / 'ai_basic_law.xml'
if not XML_PATH.exists():
    raise FileNotFoundError(f'XML 파일이 없습니다: {XML_PATH}')

root = ET.parse(XML_PATH).getroot()
print(f'XML 로드 완료: {XML_PATH}')
print(f'루트 태그: {root.tag}')

XML 로드 완료: /home/aiuser/rag_mini_pjt_template/data/ai_basic_law/ai_basic_law.xml
루트 태그: 법령


In [11]:
documents = []
CONTENT_TAGS = {'조문내용', '항내용', '호내용', '목내용'}

for unit in root.findall('.//조문단위'):
    number = unit.findtext('조문번호', '').strip()
    title = unit.findtext('조문제목', '').strip()
    parts = []
    for element in unit.iter():
        if element.tag in CONTENT_TAGS:
            text = ' '.join(''.join(element.itertext()).split())
            if text and text not in parts:
                parts.append(text)
    content = '\n'.join(parts)
    if content:
        documents.append({
            'id': len(documents) + 1,
            'article': f'제{number}조' if number else '',
            'title': title,
            'content': content,
            'source': XML_PATH.name,
        })

print(f'문서 수: {len(documents)}개')
print(documents[0] if documents else '문서가 없습니다.')

문서 수: 54개
{'id': 1, 'article': '제1조', 'title': '', 'content': '제1장 총칙', 'source': 'ai_basic_law.xml'}


In [ ]:
embedding_client = get_embedding_model()
qdrant_client = get_qdrant_client()
COLLECTION_NAME = 'ai_basic_law_articles_v1'
vectors = await embedding_client.aembed_documents([doc['content'] for doc in documents])

if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=len(vectors[0]), distance=Distance.COSINE),
    )

points = [
    PointStruct(id=doc['id'], vector=vector, payload=doc)
    for doc, vector in zip(documents, vectors)
]
qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f'{len(points)}개 문서를 Qdrant에 저장했습니다.')

In [ ]:
query = '고영향 인공지능이란 무엇인가요?'
# query = '이 법의 목적이 뭔가요?' # 조문단위로 잘 가져옴
# query = '인공지능산업 기반 조성을 위해 정부가 추진할 수 있는 지원 내용과 고영향 인공지능의 확인·영향평가 의무는 각각 무엇인가요?'
query_vector = await embedding_client.aembed_query(query)
result = qdrant_client.query_points(collection_name=COLLECTION_NAME, query=query_vector, limit=5)

for point in result.points:
    print(f"[{point.payload['article']} {point.payload['title']}] score={point.score:.4f}")
    print(point.payload['content'][:700])
    print()

[제33조 고영향 인공지능의 확인] score=0.5546
제33조(고영향 인공지능의 확인)
① 인공지능사업자는 인공지능 또는 이를 이용한 제품ㆍ서비스를 제공하는 경우 그 인공지능이 고영향 인공지능에 해당하는지에 대하여 사전에 검토하여야 하며, 필요한 경우 과학기술정보통신부장관에게 고영향 인공지능에 해당하는지 여부의 확인을 요청할 수 있다.
② 과학기술정보통신부장관은 제1항에 따른 요청이 있는 경우 고영향 인공지능 해당 여부를 확인하여야 하며, 필요한 경우 전문위원회를 설치하여 관련 자문을 받을 수 있다.
③ 과학기술정보통신부장관은 고영향 인공지능의 기준과 예시 등에 관한 가이드라인을 수립하여 보급할 수 있다.
④ 그 밖에 제1항에 따른 확인 절차 등에 관하여 필요한 사항은 대통령령으로 정한다.

[제13조 ] score=0.5460
제1절 인공지능산업 기반 조성

[제31조 인공지능 투명성 확보 의무] score=0.5278
제31조(인공지능 투명성 확보 의무)
① 인공지능사업자는 고영향 인공지능이나 생성형 인공지능을 이용한 제품 또는 서비스를 제공하려는 경우 제품 또는 서비스가 해당 인공지능에 기반하여 운용된다는 사실을 이용자에게 사전에 고지하여야 한다.
② 인공지능사업자는 생성형 인공지능 또는 이를 이용한 제품 또는 서비스를 제공하는 경우 그 결과물이 생성형 인공지능에 의하여 생성되었다는 사실을 표시하여야 한다.
③ 인공지능사업자는 인공지능시스템을 이용하여 실제와 구분하기 어려운 가상의 음향, 이미지 또는 영상 등의 결과물을 제공하는 경우 해당 결과물이 인공지능시스템에 의하여 생성되었다는 사실을 이용자가 명확하게 인식할 수 있는 방식으로 고지 또는 표시하여야 한다. 이 경우 해당 결과물이 예술적ㆍ창의적 표현물에 해당하거나 그 일부를 구성하는 경우에는 전시 또는 향유 등을 저해하지 아니하는 방식으로 고지 또는 표시할 수 있다.
④ 그 밖에 제1항에 따른 사전고지, 제2항에 따른 표시, 제3항에 따른 고지 또는 표시의 방법 및 그 예외 등에 관하여 필요한 사